<a href="https://colab.research.google.com/github/Locket51/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Locket51/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 114 (delta 28), reused 100 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 1.84 MiB | 8.21 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/flyrank-ml-internship


In [2]:
!ls data/raw/
!ls work/notebooks/

content_refresh_anonymized.csv
capstone.ipynb			 w04_baseline_score.ipynb
w01_research_question.ipynb	 w04_signal_audit.ipynb
w02_ml_task_framing.ipynb	 w05_model.ipynb
w03_data_contract.ipynb		 w06_validation_audit.ipynb
w03_feature_leakage_check.ipynb  w07_action_playbook.ipynb


In [3]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
print(df.columns.tolist())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [4]:
declining_share = (df["trend_direction"] == "down").mean()
print("share declining:", round(declining_share, 3))

declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).mean()
print("share declining AND impressions_90d >= 100:", round(declining_with_demand, 3))

print(df["impressions_90d"].describe(percentiles=[0.5, 0.9, 0.99]).round(1))

share declining: 0.542
share declining AND impressions_90d >= 100: 0.438
count     30000.0
mean       5200.4
std       16838.0
min           1.0
50%         731.0
90%       12136.4
99%       73505.8
max      517715.0
Name: impressions_90d, dtype: float64


## 1. My lane and why

**Lane 2: Refresh / Content Opportunity Scoring.**

I chose this lane because it maps directly onto the starter dataset and onto a real decision someone makes: a content reviewer with limited hours has to decide which pages to open first. In the starter slice, 54.2% of pages are already labeled as declining, and 43.8% are both declining AND have at least 100 impressions — so there is plenty of signal and plenty of real demand attached to it. That means I can spend week 1 framing the decision honestly and later weeks trying to beat a transparent baseline with a learned ranking. It is also the deepest-data lane: the guide's density table shows GSC impressions and clicks are far more common than AI sessions, so this queue is less likely to be built on noise.

## 2. The question: decision, action, cost of a wrong call

**Decision it improves:** which pages a reviewer opens first out of a large content inventory, given that no reviewer can inspect every page.

**Who acts on it:** a content or SEO reviewer with limited review capacity per week.

**Action taken from the output:** open the top-K pages from a ranked queue, read each page's reason code (e.g. stale_visible_page, declining_with_demand, low_ctr_visible_page), and decide whether to refresh, expand, protect, prune, or just monitor that page.

**Cost of a wrong recommendation:**
- False positive (page flagged that did not need work): the reviewer wastes hours on a page that was fine, and misses the pages they did not get to. Cost = reviewer time + opportunity cost.
- False negative (page missed that did need work): a page that was quietly losing visibility keeps losing it. Cost = compounding traffic loss.

Because reviewer hours are the scarce resource, this is a ranking problem, not a classification-accuracy problem. What matters is precision@K, not overall accuracy.

## 3. Quick look at the data (2-3 real numbers)

Below I load the starter CSV and print a few real numbers that make this lane worth the next 7 weeks.

In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("rows, cols:", df.shape)

declining_share = (df["trend_direction"] == "down").mean()
print(f"share of rows with trend_direction == 'down': {declining_share:.3f}")

declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).mean()
print(f"share declining AND impressions_90d >= 100: {declining_with_demand:.3f}")

print("\nimpressions_90d describe:")
print(df["impressions_90d"].describe(percentiles=[0.5, 0.9, 0.99]).round(1))

rows, cols: (30000, 44)
share of rows with trend_direction == 'down': 0.542
share declining AND impressions_90d >= 100: 0.438

impressions_90d describe:
count     30000.0
mean       5200.4
std       16838.0
min           1.0
50%         731.0
90%       12136.4
99%       73505.8
max      517715.0
Name: impressions_90d, dtype: float64


**What the numbers say:**

1. **The label is not rare.** 54.2% of rows are `trend_direction == "down"`. Unlike AI-session work (where the guide warns positives are ~30k out of ~79M), here the target class is healthy — a binary ranking problem is viable.

2. **Declining pages still carry demand.** 43.8% of rows are both declining AND have at least 100 impressions_90d, which is the floor the guide uses for its `declining_with_demand` reason code. So a queue built on this lane would point at pages that actually matter, not dead weight.

3. **Visibility is heavily skewed.** Median impressions_90d is 731, but the 99th percentile is 73,506 — a ~100x gap. A naive "sort by impressions" list would be dominated by a handful of giants. That is exactly the kind of problem a scored, reason-coded ranking is supposed to solve.

Together: the data supports a refresh-prioritization question, the label is learnable, and there is enough skew that a good ranking should beat a blunt rule.

## 4. Careful words: what I can and can't claim

**What this work will be able to say (observed, measured, directional, decision-support):**
- Observed: "In this slice, pages with signal X had outcome Y."
- Directional: "This signal is associated with a higher chance of being a good review candidate."
- Decision-support: "Ranked this way, the top-K queue put more genuinely declining pages in front of a reviewer than the baseline rule did." That is a claim about the ranking, not about the pages.
- I can compare my model to a transparent baseline using precision@K, average precision, and ROC AUC, with client-holdout validation, and report whether the learned ranking beat the rule.

**What this work will never claim:**
- That refreshing a page caused a recovery. The data has no experiment; causal claims would need a causal design.
- That I found a Google ranking factor, or that I can "predict Google."
- That a flagged page is guaranteed to improve if edited — only that it is a better candidate to review first.
- Anything about AI citations, AI rankings, or AI search visibility (not what this data measures).
- Anything built from raw queries, URLs, titles, domains, or client names.

**Language rule I will follow:** observed, measured, associated with, directional, decision-support — never proves, causes, or predicts Google.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card